# Búsqueda Semántica con FAISS y Sentence Transformers

**System:** Enterprise Vector Search & RAG Architecture  
**Objetivo:** Implementar un motor de búsqueda semántica vectorial de alta velocidad utilizando modelos de lenguaje preentrenados (*Sentence Transformers*) e indexación con *Facebook AI Similarity Search (FAISS)*.

---

## 1. Fundamentos Teóricos: Búsqueda Léxica vs. Búsqueda Semántica

En los motores de búsqueda tradicionales basados en coincidencia léxica (como BM25 o TF-IDF), la recuperación de información depende de la superposición exacta de palabras clave. Esto presenta limitaciones críticas:
- **Sinonimia:** Si un usuario busca *"café"* y el documento habla de *"granos tostados de moka"*, el motor léxico no encuentra coincidencia.
- **Polisemia y Contexto:** No distingue el significado contextual de las palabras.

La **búsqueda semántica vectorial** proyecta textos en un espacio euclídeo de alta dimensión (espacio latente). Textos con significados cercanos quedan posicionados a distancias cortas (distancia L2 o similitud coseno), permitiendo recuperar información por **significado e intención**, base fundamental de las arquitecturas **RAG (Retrieval-Augmented Generation)**.

In [1]:
# 1. Configuración del entorno e importaciones
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("Dependencias importadas correctamente. FAISS version:", faiss.__version__)

## 2. Carga del Modelo de Embeddings

Utilizamos , un modelo destilado y optimizado basado en arquitectura BERT que mapea oraciones a un espacio vectorial denso de **384 dimensiones**. Destaca por su excelente equilibrio entre velocidad de inferencia y calidad semántica.

In [2]:
# Carga del modelo preentrenado desde Hugging Face
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"Modelo cargado: {model.get_sentence_embedding_dimension()} dimensiones por vector.")

## 3. Definición del Corpus Documental y Vectorización

Definimos una base de conocimiento documental heterogénea que abarca distintos dominios semánticos (naturaleza, computación cuántica, mascotas, historia y gastronomía).

In [3]:
# Corpus de documentos
documents = [
    "El amanecer en la montaña revela colores que no existen en la ciudad.",
    "Los algoritmos cuánticos prometen cambiar la forma en que resolvemos problemas complejos.",
    "Un perro corre feliz por la playa mientras las olas rompen suavemente.",
    "La historia del arte está llena de secretos que aún no hemos descubierto.",
    "El café colombiano tiene un aroma que despierta recuerdos de infancia.",
]

# Generación de vectores latentes (embeddings)
doc_vectors = model.encode(documents)
doc_vectors = np.array(doc_vectors, dtype="float32")

print(f"Matriz de embeddings calculada: {doc_vectors.shape[0]} documentos x {doc_vectors.shape[1]} dimensiones.")

## 4. Construcción del Índice FAISS

FAISS (*Facebook AI Similarity Search*) proporciona algoritmos de indexación de vectores de alto rendimiento. En este caso utilizamos , que realiza una búsqueda exhaustiva exacta de vecinos más cercanos mediante **Distancia Euclidiana L2** ((u,v) = \sqrt{\sum (u_i - v_i)^2}$).

In [4]:
# Inicialización del índice plano L2 con la dimensionalidad requerida
dimension = doc_vectors.shape[1]
index = faiss.IndexFlatL2(dimension)

# Adición de los vectores al índice
index.add(doc_vectors)
print(f"Índice FAISS construido con éxito. Total de vectores indexados: {index.ntotal}")

## 5. Inferencia: Consulta Semántica y Recuperación Top-K

Formulamos una consulta en lenguaje natural con matices emocionales y semánticos para evaluar la capacidad de emparejamiento conceptual.

In [5]:
# Consulta del usuario
query = "¿A qué me recuerda cundo me tomo un café colombiano?"

# Codificación vectorial de la consulta
query_vector = model.encode([query])
query_vector = np.array(query_vector, dtype="float32")

# Búsqueda en el índice de los k vecinos más cercanos (k=1)
k = 1
distances, indices = index.search(query_vector, k=k)
nearest_index = indices[0][0]

# Visualización de resultados
print("
" + "="*60)
print("  BÚSQUEDA SEMÁNTICA CON FAISS")
print("="*60)
print(f"
  Consulta formulada:  {query}")
print(f"  Documento recuperado: {documents[nearest_index]}")
print(f"  Distancia L2:        {distances[0][0]:.4f}")
print(f"  Índice documental:   {nearest_index}")
print("="*60)

  BÚSQUEDA SEMÁNTICA CON FAISS

  Consulta formulada:  ¿A qué me recuerda cundo me tomo un café colombiano?
  Documento recuperado: El café colombiano tiene un aroma que despierta recuerdos de infancia.
  Distancia L2:        0.5841
  Índice documental:   4


## 6. Conclusiones y Escalabilidad en Producción

1. **Efectividad Semántica:** A pesar de que la consulta utiliza una formulación interrogativa y gramática libre, el modelo asocia con precisión el concepto con el documento 4 mediante proximidad vectorial.
2. **Escalabilidad:** Para millones de documentos en entornos empresariales, el índice  puede sustituirse por  (Voronoi inverted file) o  (Hierarchical Navigable Small World) para búsqueda aproximada (ANN) en microsegundos.